# Breast Cancer Classification — Train, Evaluate, Save

**Run this notebook in Jupyter Notebook or Google Colab, not VS Code.**

1. Download `data.csv` from the Kaggle dataset linked in the README.
2. Upload it to the same folder as this notebook (in Colab, upload it to the current runtime).
3. Run every cell in order.
4. This notebook writes `model.joblib` and `feature_names.json` into the current working directory. Copy both files into the VS Code project root after training in Colab.

Educational demo only; not for medical diagnosis.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = Path("data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "data.csv not found. Download it from Kaggle and upload/place it beside this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Clean column names and remove empty unnamed columns.
df.columns = [str(c).strip() for c in df.columns]
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

if "diagnosis" not in df.columns:
    raise ValueError("Expected a 'diagnosis' target column. Check that you downloaded the Kaggle WDBC data.csv.")

# The dataset uses B = benign and M = malignant.
df["diagnosis"] = df["diagnosis"].astype(str).str.strip().str.upper()
df = df[df["diagnosis"].isin(["B", "M"])].copy()

# Drop identifiers; keep the diagnosis target separate.
drop_columns = [c for c in ["id", "ID", "diagnosis"] if c in df.columns]
X = df.drop(columns=drop_columns + (["diagnosis"] if "diagnosis" not in drop_columns else []), errors="ignore")
y = df["diagnosis"]

# Convert features to numeric and remove any columns that contain no numeric values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1, how="all")
# Median imputation for any remaining missing values.
X = X.fillna(X.median(numeric_only=True))

if X.shape[1] != 30:
    print(f"Warning: found {X.shape[1]} feature columns rather than the expected 30. Verify the dataset version.")

feature_names = X.columns.tolist()
print("Number of model features:", len(feature_names))
print("Class distribution:")
display(y.value_counts().rename_axis("diagnosis").to_frame("count"))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test precision (M positive): {precision_score(y_test, y_pred, pos_label='M'):.4f}")
print(f"Test recall (M positive):    {recall_score(y_test, y_pred, pos_label='M'):.4f}")
print(f"Test F1 (M positive):        {f1_score(y_test, y_pred, pos_label='M'):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=["B", "M"], target_names=["Benign (B)", "Malignant (M)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benign (B)", "Malignant (M)"])
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Held-out Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model and exact feature order for the API.
joblib.dump(model, "model.joblib")
Path("feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

print("Saved model.joblib")
print("Saved feature_names.json")
print("Feature count:", len(feature_names))
print("\nCopy these two files into the root of your VS Code project.")
